# 13 — Star Schema Export (Fact & Dimension Tables)

**Tujuan notebook ini:**
Bangun ulang data mart sebagai **star schema** proper — fact table di tengah, dimension table di sekitarnya — supaya semua dashboard Power BI bisa saling cross-filter penuh (customer ↔ produk ↔ seller ↔ tanggal ↔ geolokasi).

**Fact Tables:**
- `Fact_Orders` — grain: 1 order
- `Fact_Order_Items` — grain: 1 baris produk dalam 1 order (punya `customer_unique_id` didenormalisasi sebagai jembatan)
- `Fact_Payments` — grain: 1 payment record

**Dimension Tables:**
- `Dim_Customer`, `Dim_Product`, `Dim_Seller`, `Dim_Date`, `Dim_Geolocation`


In [1]:
import sys
try:
    import pyarrow
except ModuleNotFoundError:
    !{sys.executable} -m pip install pyarrow

import pandas as pd
import numpy as np
from pymongo import MongoClient
from pathlib import Path

pd.set_option("display.max_columns", None)

DATA_DIR = Path("../data_processed")   # sumber: CSV hasil notebook 04-12 (staging/eksplorasi)
MART_DIR = Path("../data_mart")        # tujuan: 8 tabel star schema final, siap Power BI
MART_DIR.mkdir(exist_ok=True)

MONGO_URI = "mongodb://localhost:27017/"
client = MongoClient(MONGO_URI)
db = client["olist_db"]

# Load semua raw collection
customers_raw = pd.DataFrame(list(db["customers_raw"].find({}, {"_id": 0})))
orders_raw = pd.DataFrame(list(db["orders_raw"].find({}, {"_id": 0})))
order_items_raw = pd.DataFrame(list(db["order_items_raw"].find({}, {"_id": 0})))
payments_raw = pd.DataFrame(list(db["payments_raw"].find({}, {"_id": 0})))
reviews_raw = pd.DataFrame(list(db["reviews_raw"].find({}, {"_id": 0})))
products_raw = pd.DataFrame(list(db["products_raw"].find({}, {"_id": 0})))
sellers_raw = pd.DataFrame(list(db["sellers_raw"].find({}, {"_id": 0})))
geolocation_raw = pd.DataFrame(list(db["geolocation_raw"].find({}, {"_id": 0})))
category_translation_raw = pd.DataFrame(list(db["category_translation_raw"].find({}, {"_id": 0})))

# Load hasil analytics/ML yang sudah ada (untuk atribut Dim_Customer)
customer_segmentation = pd.read_csv(DATA_DIR / "customer_segmentation_full.csv")
predictive_analytics = pd.read_csv(DATA_DIR / "dashboard5_predictive_analytics.csv")

print("Semua sumber data ter-load.")


Semua sumber data ter-load.


---
## 1. `Dim_Date`

Dibangun dari rentang tanggal dataset — 1 baris per tanggal kalender, lengkap dengan atribut turunan (tahun, bulan, kuartal, hari dalam minggu).


In [2]:
orders_raw["order_purchase_timestamp"] = pd.to_datetime(orders_raw["order_purchase_timestamp"], errors="coerce")

date_min = orders_raw["order_purchase_timestamp"].min().normalize()
date_max = orders_raw["order_purchase_timestamp"].max().normalize()

Dim_Date = pd.DataFrame({"date_key": pd.date_range(date_min, date_max, freq="D")})
Dim_Date["year"] = Dim_Date["date_key"].dt.year
Dim_Date["month"] = Dim_Date["date_key"].dt.month
Dim_Date["month_name"] = Dim_Date["date_key"].dt.month_name()
Dim_Date["quarter"] = Dim_Date["date_key"].dt.quarter
Dim_Date["week"] = Dim_Date["date_key"].dt.isocalendar().week.astype(int)
Dim_Date["day_of_week"] = Dim_Date["date_key"].dt.dayofweek
Dim_Date["day_name"] = Dim_Date["date_key"].dt.day_name()
Dim_Date["is_weekend"] = Dim_Date["day_of_week"].isin([5, 6])
Dim_Date["year_month"] = Dim_Date["date_key"].dt.to_period("M").astype(str)

print(f"Dim_Date: {Dim_Date.shape[0]:,} baris ({date_min.date()} s.d. {date_max.date()})")
Dim_Date.head()


Dim_Date: 774 baris (2016-09-04 s.d. 2018-10-17)


,date_key,year,month,month_name,quarter,week,day_of_week,day_name,is_weekend,year_month
0,2016-09-04,2016,9,September,3,35,6,Sunday,True,2016-09
1,2016-09-05,2016,9,September,3,36,0,Monday,False,2016-09
2,2016-09-06,2016,9,September,3,36,1,Tuesday,False,2016-09
3,2016-09-07,2016,9,September,3,36,2,Wednesday,False,2016-09
4,2016-09-08,2016,9,September,3,36,3,Thursday,False,2016-09


---
## 2. `Dim_Geolocation`

Sama seperti agregasi di Fase 3.4 — 1 baris per zip code prefix.


In [3]:
Dim_Geolocation = geolocation_raw.groupby("geolocation_zip_code_prefix").agg(
    latitude=("geolocation_lat", "median"),
    longitude=("geolocation_lng", "median"),
    city=("geolocation_city", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    state=("geolocation_state", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
).reset_index().rename(columns={"geolocation_zip_code_prefix": "zip_code_prefix"})

print(f"Dim_Geolocation: {Dim_Geolocation.shape[0]:,} baris")
Dim_Geolocation.head()


Dim_Geolocation: 19,015 baris


,zip_code_prefix,latitude,longitude,city,state
0,1001,-23.550381,-46.634027,sao paulo,SP
1,1002,-23.548551,-46.635072,sao paulo,SP
2,1003,-23.548977,-46.635313,sao paulo,SP
3,1004,-23.549535,-46.634771,sao paulo,SP
4,1005,-23.549612,-46.636532,sao paulo,SP


---
## 3. `Dim_Customer`

1 baris per `customer_unique_id`. Menggabungkan atribut deskriptif dasar + label hasil analisis (segment dari Fase 6, prediction dari Fase 10) sebagai atribut customer — bukan re-agregasi ulang total_orders/total_spending di sini (itu sebaiknya dihitung via measure DAX di Power BI dari `Fact_Orders`, bukan didenormalisasi ke dimension).


In [4]:
Dim_Customer = customers_raw[["customer_unique_id", "customer_zip_code_prefix", "customer_state"]].drop_duplicates(
    subset="customer_unique_id"
).rename(columns={"customer_zip_code_prefix": "zip_code_prefix"})

Dim_Customer = Dim_Customer.merge(
    customer_segmentation[["customer_unique_id", "segment"]],
    on="customer_unique_id", how="left"
)

Dim_Customer = Dim_Customer.merge(
    predictive_analytics[[
        "customer_unique_id", "repeat_purchase_probability", "prediction",
        "value_band", "priority_quadrant", "threshold_method"
    ]],
    on="customer_unique_id", how="left"
)

# actual_repeat: GROUND TRUTH dari observation/prediction window (Fase 7) - WAJIB untuk
# Precision@Top-K & Lift yang dihitung dinamis via DAX di Power BI. Tanpa ini, Power BI
# cuma bisa nampilkan 'prediction' model, tidak bisa validasi apakah prediction itu benar.
ml_dataset_target = pd.read_csv(DATA_DIR / "ml_dataset_observation.csv")[
    ["customer_unique_id", "target"]
].rename(columns={"target": "actual_repeat"})
ml_dataset_target["actual_repeat"] = ml_dataset_target["actual_repeat"].astype(bool)

Dim_Customer = Dim_Customer.merge(ml_dataset_target, on="customer_unique_id", how="left")

print(f"Dim_Customer: {Dim_Customer.shape[0]:,} baris")
print(f"Customer dengan prediction (eligible di observation window): {Dim_Customer['prediction'].notnull().sum():,}")
print(f"Customer TANPA prediction (di luar eligible population): {Dim_Customer['prediction'].isnull().sum():,}")
print(f"Customer dengan actual_repeat (ground truth tersedia): {Dim_Customer['actual_repeat'].notnull().sum():,}")
Dim_Customer.head()


Dim_Customer: 96,096 baris
Customer dengan prediction (eligible di observation window): 54,738
Customer TANPA prediction (di luar eligible population): 41,358
Customer dengan actual_repeat (ground truth tersedia): 54,738


,customer_unique_id,zip_code_prefix,customer_state,segment,repeat_purchase_probability,prediction,value_band,priority_quadrant,threshold_method,actual_repeat
0,861eff4711a542e4b93843c6dd7febb0,14409,SP,Low Value,0.413350,0.0,High Value,WIN-BACK,top_5_percent,False
1,290c77bc529b7ac935b93aa66c333dc3,9790,SP,Low Value,0.497619,0.0,High Value,WIN-BACK,top_5_percent,False
2,060e732b5b29e8181a18229c7b0b2b5e,1151,SP,Low Value,NaN,NaN,NaN,NaN,NaN,NaN
3,259dac757896d24d7702b9acbbff3f3c,8775,SP,Low Value,NaN,NaN,NaN,NaN,NaN,NaN
4,345ecd01c38d18a9036ed96c73b8d066,13056,SP,Low Value,NaN,NaN,NaN,NaN,NaN,NaN


> **Catatan penting:** kolom `segment`, `repeat_purchase_probability`, `prediction`, `value_band`, `priority_quadrant`, dan **`actual_repeat`** akan **NULL** untuk customer yang tidak eligible di observation window Fase 7 (order pertamanya di luar rentang training). Ini bukan data hilang — ini sinyal populasi yang memang tidak masuk scope scoring, dan harus ditangani eksplisit saat bikin visual/DAX di Power BI (filter `NOT(ISBLANK(prediction))` sebelum menghitung Precision@Top-K).
>
> **`actual_repeat`** adalah ground truth (apakah customer BENERAN melakukan repeat order di prediction window) — dipakai khusus untuk validasi model secara dinamis di Power BI (Precision@Top-K, Lift). Ini BUKAN informasi yang tersedia di dunia nyata saat scoring customer baru (baru diketahui setelah prediction window berlalu) — kolom ini ada di data mart untuk keperluan evaluasi model, bukan untuk dipakai sebagai fitur real-time.


---
## 4. `Dim_Product`


In [5]:
Dim_Product = products_raw.merge(
    category_translation_raw, on="product_category_name", how="left"
)[["product_id", "product_category_name_english"]].rename(
    columns={"product_category_name_english": "category"}
).drop_duplicates(subset="product_id")

print(f"Dim_Product: {Dim_Product.shape[0]:,} baris")
Dim_Product.head()


Dim_Product: 32,951 baris


,product_id,category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,art
2,96bd76ec8810374ed1b65e291975717f,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,baby
4,9dc1a7de274444849c219cff195d0b71,housewares


---
## 5. `Dim_Seller`


In [6]:
Dim_Seller = sellers_raw[["seller_id", "seller_zip_code_prefix", "seller_state"]].drop_duplicates(
    subset="seller_id"
).rename(columns={"seller_zip_code_prefix": "zip_code_prefix"})

print(f"Dim_Seller: {Dim_Seller.shape[0]:,} baris")
Dim_Seller.head()


Dim_Seller: 3,095 baris


,seller_id,zip_code_prefix,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,SP


---
## 6. `Fact_Orders`

Grain: 1 baris = 1 order.


In [7]:
orders = orders_raw.copy()
for col in ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders = orders.merge(customers_raw[["customer_id", "customer_unique_id"]], on="customer_id", how="left")

transaction_value_per_order = payments_raw.groupby("order_id")["payment_value"].sum().rename("transaction_value")
orders = orders.merge(transaction_value_per_order, on="order_id", how="left")

review_per_order = reviews_raw.groupby("order_id")["review_score"].mean().rename("review_score")
orders = orders.merge(review_per_order, on="order_id", how="left")

n_items_per_order = order_items_raw.groupby("order_id")["order_item_id"].count().rename("n_items")
orders = orders.merge(n_items_per_order, on="order_id", how="left")

orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days
orders["is_delivered"] = orders["order_delivered_customer_date"].notnull()
orders["order_purchase_date"] = orders["order_purchase_timestamp"].dt.normalize()

Fact_Orders = orders[[
    "order_id", "customer_unique_id", "order_purchase_date", "order_status",
    "transaction_value", "review_score", "n_items", "delivery_days", "is_delivered",
]].copy()

print(f"Fact_Orders: {Fact_Orders.shape[0]:,} baris")
Fact_Orders.head()


Fact_Orders: 99,441 baris


,order_id,customer_unique_id,order_purchase_date,order_status,transaction_value,review_score,n_items,delivery_days,is_delivered
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,2017-10-02,delivered,38.71,4.0,1.0,8.0,True
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,2018-07-24,delivered,141.46,4.0,1.0,13.0,True
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,2018-08-08,delivered,179.12,5.0,1.0,9.0,True
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,2017-11-18,delivered,72.20,5.0,1.0,13.0,True
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,2018-02-13,delivered,28.62,5.0,1.0,2.0,True


---
## 7. `Fact_Order_Items`

Grain: 1 baris = 1 produk di dalam 1 order. `customer_unique_id` dan `order_purchase_date` didenormalisasi langsung ke sini — inilah tabel **jembatan** yang menghubungkan `Dim_Customer`, `Dim_Product`, `Dim_Seller`, dan `Dim_Date` sekaligus, supaya cross-filter antar dashboard customer/produk/seller bisa jalan.


In [8]:
Fact_Order_Items = order_items_raw.merge(
    orders[["order_id", "customer_unique_id", "order_purchase_date"]], on="order_id", how="left"
)[[
    "order_id", "order_item_id", "product_id", "seller_id",
    "customer_unique_id", "order_purchase_date", "price", "freight_value",
]]

print(f"Fact_Order_Items: {Fact_Order_Items.shape[0]:,} baris")
Fact_Order_Items.head()


Fact_Order_Items: 112,650 baris


,order_id,order_item_id,product_id,seller_id,customer_unique_id,order_purchase_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,871766c5855e863f6eccc05f988b23cb,2017-09-13,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,eb28e67c4c0b83846050ddfb8a35d051,2017-04-26,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,3818d81c6709e39d06b2738a8d3a2474,2018-01-14,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,af861d436cfc08b2c2ddefd0ba074622,2018-08-08,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,64b576fb70d441e8f1b2d7d446e483c5,2017-02-04,199.90,18.14


---
## 8. `Fact_Payments`

Grain: 1 baris = 1 payment record (order bisa punya banyak, misal cicilan).


In [9]:
Fact_Payments = payments_raw.merge(
    orders[["order_id", "customer_unique_id", "order_purchase_date"]], on="order_id", how="left"
)[[
    "order_id", "payment_sequential", "customer_unique_id", "order_purchase_date",
    "payment_type", "payment_installments", "payment_value",
]]

print(f"Fact_Payments: {Fact_Payments.shape[0]:,} baris")
Fact_Payments.head()


Fact_Payments: 103,886 baris


,order_id,payment_sequential,customer_unique_id,order_purchase_date,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,708ab75d2a007f0564aedd11139c7708,2018-04-25,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,a8b9d3a27068454b1c98cc67d4e31e6f,2018-06-26,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,6f70c0b2f7552832ba46eb57b1c5651e,2017-12-12,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,87695ed086ebd36f20404c82d20fca87,2017-12-06,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,4291db0da71914754618cd789aebcd56,2018-05-21,credit_card,2,128.45


---
## 9. Export Semua Tabel ke Parquet


In [10]:
star_schema_tables = {
    "Dim_Customer": Dim_Customer,
    "Dim_Product": Dim_Product,
    "Dim_Seller": Dim_Seller,
    "Dim_Date": Dim_Date,
    "Dim_Geolocation": Dim_Geolocation,
    "Fact_Orders": Fact_Orders,
    "Fact_Order_Items": Fact_Order_Items,
    "Fact_Payments": Fact_Payments,
}

for name, df in star_schema_tables.items():
    output_path = MART_DIR / f"{name}.parquet"
    df.to_parquet(output_path, engine="pyarrow", index=False)
    print(f"{name}: {df.shape[0]:,} baris, {df.shape[1]} kolom -> {output_path}")


Dim_Customer: 96,096 baris, 10 kolom -> ..\data_mart\Dim_Customer.parquet
Dim_Product: 32,951 baris, 2 kolom -> ..\data_mart\Dim_Product.parquet
Dim_Seller: 3,095 baris, 3 kolom -> ..\data_mart\Dim_Seller.parquet
Dim_Date: 774 baris, 10 kolom -> ..\data_mart\Dim_Date.parquet
Dim_Geolocation: 19,015 baris, 5 kolom -> ..\data_mart\Dim_Geolocation.parquet
Fact_Orders: 99,441 baris, 9 kolom -> ..\data_mart\Fact_Orders.parquet
Fact_Order_Items: 112,650 baris, 8 kolom -> ..\data_mart\Fact_Order_Items.parquet
Fact_Payments: 103,886 baris, 7 kolom -> ..\data_mart\Fact_Payments.parquet


---
## 10. Validasi Grain (Wajib Sebelum Dipakai di Power BI)

Pastikan tiap fact/dimension table benar-benar sesuai grain yang didefinisikan — cek duplicate key.


In [11]:
grain_checks = [
    ("Dim_Customer", Dim_Customer, "customer_unique_id"),
    ("Dim_Product", Dim_Product, "product_id"),
    ("Dim_Seller", Dim_Seller, "seller_id"),
    ("Dim_Date", Dim_Date, "date_key"),
    ("Dim_Geolocation", Dim_Geolocation, "zip_code_prefix"),
    ("Fact_Orders", Fact_Orders, "order_id"),
]

for name, df, key in grain_checks:
    dup_count = df[key].duplicated().sum()
    status = "✅" if dup_count == 0 else "⚠️"
    print(f"{status} {name}: duplicate '{key}' = {dup_count}")

# Fact_Order_Items & Fact_Payments punya grain gabungan (bukan 1 kolom saja)
foi_dup = Fact_Order_Items.duplicated(subset=["order_id", "order_item_id"]).sum()
print(f"{'✅' if foi_dup == 0 else '⚠️'} Fact_Order_Items: duplicate (order_id, order_item_id) = {foi_dup}")

fp_dup = Fact_Payments.duplicated(subset=["order_id", "payment_sequential"]).sum()
print(f"{'✅' if fp_dup == 0 else '⚠️'} Fact_Payments: duplicate (order_id, payment_sequential) = {fp_dup}")


✅ Dim_Customer: duplicate 'customer_unique_id' = 0
✅ Dim_Product: duplicate 'product_id' = 0
✅ Dim_Seller: duplicate 'seller_id' = 0
✅ Dim_Date: duplicate 'date_key' = 0
✅ Dim_Geolocation: duplicate 'zip_code_prefix' = 0
✅ Fact_Orders: duplicate 'order_id' = 0
✅ Fact_Order_Items: duplicate (order_id, order_item_id) = 0
✅ Fact_Payments: duplicate (order_id, payment_sequential) = 0


---
## Definition of Done

- [ ] 5 dimension table dibangun dengan grain yang benar (1 baris per entitas)
- [ ] 3 fact table dibangun dengan grain yang benar, tervalidasi tidak ada duplicate key
- [ ] `Fact_Order_Items` punya `customer_unique_id` terdenormalisasi sebagai jembatan cross-filter
- [ ] Semua 8 tabel tersimpan sebagai `.parquet` di `data_mart/` (terpisah dari `data_processed/`)
- [ ] Dokumentasi star schema (relationship diagram, cardinality) tersedia di `docs/star_schema.md`

**Lanjut ke:** import 8 tabel ini ke Power BI, bangun relasi sesuai `docs/star_schema.md`.
